# Long Puts vs. Put Spreads
### Regime-Conditioned Backtest — VIX × MOVE × COR1M Signal Framework

---

**Hypothesis:** The optimal options hedging structure is regime-dependent. Long puts offer uncapped tail protection but carry higher theta drag. Put spreads reduce premium cost but impose a profit ceiling — one that may be breached precisely during the worst drawdowns.

This notebook compares both structures across volatility regimes defined by the combined VIX × MOVE × COR1M signal matrix.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.ticker as mticker
from matplotlib.patches import Patch
import warnings
warnings.filterwarnings('ignore')

# ── Palette ───────────────────────────────────────────────────────────────────
FID_GREEN       = '#00843D'
FID_GREEN_LIGHT = '#5BAD7F'
FID_NAVY        = '#005EB8'
FID_CHARCOAL    = '#1A1A1A'
FID_GRAY_DARK   = '#4A4A4A'
FID_GRAY_MID    = '#767676'
FID_GRAY_LIGHT  = '#E8E8E8'
FID_RULE        = '#CCCCCC'
FID_WHITE       = '#FFFFFF'
FID_AMBER       = '#B8860B'
FID_RED         = '#C0392B'
FID_RED_DARK    = '#8B0000'

LP_COLOR = FID_GREEN   # Long Put
PS_COLOR = FID_NAVY    # Put Spread

REGIME_COLORS = {
    'FULL DEPLOYMENT':    FID_GREEN,
    'STANDARD OPS':       FID_GREEN_LIGHT,
    'REDUCE SIZING':      FID_AMBER,
    'REDUCE / BUILD LIST':'#C8882A',
    'HIGH STRESS':        FID_RED,
    'QUALITY ONLY':       '#A93226',
    'MAX CASH':           FID_RED_DARK,
    'CRISIS':             '#6B0000',
    'EXTREME CRISIS':     '#4A0000',
    'UNKNOWN':            '#AAAAAA',
}

# ── Global style ──────────────────────────────────────────────────────────────
plt.rcParams.update({
    'figure.facecolor':  FID_WHITE,
    'axes.facecolor':    FID_WHITE,
    'axes.edgecolor':    FID_RULE,
    'axes.labelcolor':   FID_CHARCOAL,
    'axes.spines.top':   False,
    'axes.spines.right': False,
    'xtick.color':       FID_GRAY_MID,
    'ytick.color':       FID_GRAY_MID,
    'xtick.labelsize':   9,
    'ytick.labelsize':   9,
    'text.color':        FID_CHARCOAL,
    'grid.color':        FID_GRAY_LIGHT,
    'grid.linewidth':    0.6,
    'font.family':       'sans-serif',
    'font.sans-serif':   ['Helvetica Neue', 'Helvetica', 'Arial', 'DejaVu Sans'],
    'axes.titlesize':    11,
    'axes.titleweight':  'bold',
    'axes.titlecolor':   FID_CHARCOAL,
    'axes.labelsize':    9,
    'legend.fontsize':   8,
    'legend.framealpha': 0.95,
    'legend.edgecolor':  FID_RULE,
    'figure.dpi':        130,
    'savefig.dpi':       150,
    'savefig.bbox':      'tight',
    'savefig.facecolor': FID_WHITE,
})

def fmt_dollar(x, pos=None): return f'${x:,.0f}'
def fmt_pct(x, pos=None):    return f'{x:.0f}%'

print('Environment ready.')

---
## 1. Load and Classify Regime Data

In [ ]:
from src.backtester.engine import load_regime_data, run_backtest, trades_to_df

regime_df = load_regime_data()

print(f'Date range   : {regime_df.date.min().date()}  to  {regime_df.date.max().date()}')
print(f'Trading days : {len(regime_df):,}')
print()
print('Regime distribution')
print('─' * 42)
counts = regime_df.combined.value_counts()
for regime, n in counts.items():
    pct = n / len(regime_df) * 100
    print(f'  {regime:<30}  {n:>4}  ({pct:.1f}%)')

---
## 2. Regime Timeline

In [ ]:
fig, axes = plt.subplots(4, 1, figsize=(14, 9), sharex=True,
                          gridspec_kw={'height_ratios': [3, 1.4, 1.4, 1.4], 'hspace': 0.08})

fig.text(0.02, 0.98, 'Regime Dashboard — Full History',
         fontsize=13, fontweight='bold', va='top')

dates = pd.to_datetime(regime_df['date'])

ax = axes[0]
for regime, color in REGIME_COLORS.items():
    mask = regime_df['combined'] == regime
    if mask.any():
        ax.fill_between(dates, regime_df['spot'].min() * 0.97,
                         regime_df['spot'].max() * 1.02,
                         where=mask, alpha=0.12, color=color)
ax.plot(dates, regime_df['spot'], color=FID_CHARCOAL, linewidth=1.1)
ax.set_ylabel('SPY')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(fmt_dollar))
ax.grid(True, axis='y')
patches = [Patch(color=c, alpha=0.6, label=r)
           for r, c in REGIME_COLORS.items() if r in regime_df['combined'].values]
ax.legend(handles=patches, loc='upper left', ncol=4, fontsize=7, edgecolor=FID_RULE)

ax2 = axes[1]
ax2.plot(dates, regime_df['vix'], color=FID_CHARCOAL, linewidth=0.9)
ax2.axhline(25, color=FID_GREEN, linestyle='--', linewidth=0.8, label='25')
ax2.axhline(35, color=FID_RED,   linestyle='--', linewidth=0.8, label='35')
ax2.set_ylabel('VIX')
ax2.legend(loc='upper right', fontsize=7, title='Thresholds', title_fontsize=7)
ax2.grid(True, axis='y')

ax3 = axes[2]
ax3.plot(dates, regime_df['move'], color=FID_GRAY_DARK, linewidth=0.9)
ax3.axhline(90,  color=FID_GREEN, linestyle='--', linewidth=0.8, label='90')
ax3.axhline(120, color=FID_RED,   linestyle='--', linewidth=0.8, label='120')
ax3.set_ylabel('MOVE')
ax3.legend(loc='upper right', fontsize=7, title='Thresholds', title_fontsize=7)
ax3.grid(True, axis='y')

ax4 = axes[3]
ax4.plot(dates, regime_df['cor1m'], color=FID_GRAY_MID, linewidth=0.9)
ax4.axhline(40, color=FID_RED, linestyle='--', linewidth=0.8, label='40')
ax4.set_ylabel('COR1M')
ax4.legend(loc='upper right', fontsize=7, title='Threshold', title_fontsize=7)
ax4.grid(True, axis='y')

for ax in axes:
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
    ax.xaxis.set_major_locator(mdates.YearLocator())

plt.savefig('../results/01_regime_timeline.png')
plt.show()

---
## 3. Run Backtest

In [ ]:
lp_trades, ps_trades = run_backtest(regime_df)

lp_df = trades_to_df(lp_trades)
ps_df = trades_to_df(ps_trades)

os.makedirs('../results', exist_ok=True)
lp_df.to_csv('../results/long_put_trades.csv', index=False)
ps_df.to_csv('../results/put_spread_trades.csv', index=False)

print(f'Long Put trades   : {len(lp_df)}')
print(f'Put Spread trades : {len(ps_df)}')

---
## 4. Overall Performance

In [ ]:
from src.backtester.metrics import compute_metrics, compare_strategies

lp_m = compute_metrics(lp_df, 'Long Put')
ps_m = compute_metrics(ps_df, 'Put Spread')

summary = pd.DataFrame([lp_m, ps_m]).set_index('label')
display_cols = ['n_trades','total_pnl','total_premium','win_rate','avg_pnl','avg_premium','avg_rop','sharpe']
available = [c for c in display_cols if c in summary.columns]
print(summary[available].T.to_string())

---
## 5. Cumulative P&L

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 7), gridspec_kw={'hspace': 0.42})

fig.text(0.02, 0.98, 'Cumulative P&L — Long Put vs. Put Spread',
         fontsize=13, fontweight='bold', va='top')

lp_cum = lp_df.set_index('close_date')['pnl'].sort_index().cumsum()
ps_cum = ps_df.set_index('close_date')['pnl'].sort_index().cumsum()

ax = axes[0]
ax.plot(pd.to_datetime(lp_cum.index), lp_cum.values,
        color=LP_COLOR, linewidth=1.6, label='Long Put')
ax.plot(pd.to_datetime(ps_cum.index), ps_cum.values,
        color=PS_COLOR, linewidth=1.6, label='Put Spread')
ax.axhline(0, color=FID_RULE, linewidth=0.8)
ax.set_title('Cumulative Hedge P&L (net of premium paid)', pad=8)
ax.set_ylabel('Cumulative P&L ($)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(fmt_dollar))
ax.legend()
ax.grid(True, axis='y')

lp_prem = lp_df.set_index('close_date')['premium_paid'].sort_index().cumsum()
ps_prem = ps_df.set_index('close_date')['premium_paid'].sort_index().cumsum()

ax2 = axes[1]
ax2.plot(pd.to_datetime(lp_prem.index), lp_prem.values,
         color=LP_COLOR, linewidth=1.6, label='Long Put')
ax2.plot(pd.to_datetime(ps_prem.index), ps_prem.values,
         color=PS_COLOR, linewidth=1.6, label='Put Spread')
ax2.set_title('Cumulative Premium Paid — Cost Drag Comparison', pad=8)
ax2.set_ylabel('Cumulative Premium ($)')
ax2.yaxis.set_major_formatter(mticker.FuncFormatter(fmt_dollar))
ax2.legend()
ax2.grid(True, axis='y')

plt.savefig('../results/02_cumulative_pnl.png')
plt.show()

---
## 6. Regime-Bucketed Performance

In [ ]:
from src.backtester.metrics import compare_by_regime

regime_compare = compare_by_regime(lp_df, ps_df)
key_metrics = ['lp_n_trades','lp_total_pnl','lp_avg_rop','lp_win_rate',
               'ps_n_trades','ps_total_pnl','ps_avg_rop','ps_win_rate']
available = [c for c in key_metrics if c in regime_compare.columns]
print(regime_compare[available].to_string())

In [ ]:
regimes = regime_compare.index.tolist()
x = np.arange(len(regimes))
w = 0.36

fig, axes = plt.subplots(1, 2, figsize=(14, 5), gridspec_kw={'wspace': 0.38})

fig.text(0.02, 1.01, 'Performance by Regime — Long Put vs. Put Spread',
         fontsize=13, fontweight='bold', va='top')

ax = axes[0]
lp_pnl = regime_compare.get('lp_avg_pnl', pd.Series(0, index=regimes)).fillna(0)
ps_pnl = regime_compare.get('ps_avg_pnl', pd.Series(0, index=regimes)).fillna(0)
ax.bar(x - w/2, lp_pnl, w, label='Long Put',   color=LP_COLOR, alpha=0.9)
ax.bar(x + w/2, ps_pnl, w, label='Put Spread', color=PS_COLOR, alpha=0.9)
ax.axhline(0, color=FID_RULE, linewidth=0.8)
ax.set_xticks(x)
ax.set_xticklabels(regimes, rotation=30, ha='right', fontsize=8)
ax.set_ylabel('Avg P&L per Trade ($)')
ax.set_title('Average P&L per Trade', pad=8)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(fmt_dollar))
ax.legend()
ax.grid(True, axis='y')

ax2 = axes[1]
lp_rop = regime_compare.get('lp_avg_rop', pd.Series(0, index=regimes)).fillna(0)
ps_rop = regime_compare.get('ps_avg_rop', pd.Series(0, index=regimes)).fillna(0)
ax2.bar(x - w/2, lp_rop * 100, w, label='Long Put',   color=LP_COLOR, alpha=0.9)
ax2.bar(x + w/2, ps_rop * 100, w, label='Put Spread', color=PS_COLOR, alpha=0.9)
ax2.axhline(0, color=FID_RULE, linewidth=0.8)
ax2.set_xticks(x)
ax2.set_xticklabels(regimes, rotation=30, ha='right', fontsize=8)
ax2.set_ylabel('Return on Premium (%)')
ax2.set_title('Return on Premium by Regime', pad=8)
ax2.yaxis.set_major_formatter(mticker.FuncFormatter(fmt_pct))
ax2.legend()
ax2.grid(True, axis='y')

plt.savefig('../results/03_regime_comparison.png')
plt.show()

---
## 7. Annualized Cost Per Hedge

In [ ]:
from config import BACKTEST_START, BACKTEST_END
years = (pd.Timestamp(BACKTEST_END) - pd.Timestamp(BACKTEST_START)).days / 365.25

lp_annual = lp_df.groupby('regime')['premium_paid'].sum() / years
ps_annual = ps_df.groupby('regime')['premium_paid'].sum() / years
all_regimes = sorted(set(lp_annual.index) | set(ps_annual.index))
x = np.arange(len(all_regimes))
w = 0.36

lp_vals = [lp_annual.get(r, 0) for r in all_regimes]
ps_vals = [ps_annual.get(r, 0) for r in all_regimes]

fig, ax = plt.subplots(figsize=(12, 5))
fig.text(0.02, 0.98, 'Annualized Premium Spend by Regime',
         fontsize=13, fontweight='bold', va='top')

ax.bar(x - w/2, lp_vals, w, label='Long Put',   color=LP_COLOR, alpha=0.9)
ax.bar(x + w/2, ps_vals, w, label='Put Spread', color=PS_COLOR, alpha=0.9)

for i, (lv, pv) in enumerate(zip(lp_vals, ps_vals)):
    if lv > 0:
        savings = (lv - pv) / lv * 100
        ax.text(i, max(lv, pv) * 1.02, f'-{savings:.0f}%',
                ha='center', fontsize=8, color=FID_GRAY_DARK)

ax.set_xticks(x)
ax.set_xticklabels(all_regimes, rotation=30, ha='right')
ax.set_ylabel('Annualized Premium ($)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(fmt_dollar))
ax.legend()
ax.grid(True, axis='y')

plt.savefig('../results/04_cost_per_hedge.png')
plt.show()

---
## 8. Short Strike Breach — When the Cap Binds

In [ ]:
from config import SHORT_STRIKE_PCT

ps_analysis = ps_df.copy()
ps_analysis['spot_decline_pct'] = (
    (ps_analysis['open_spot'] - ps_analysis['close_spot']) / ps_analysis['open_spot']
)
threshold = 1 - SHORT_STRIKE_PCT
ps_analysis['cap_hit'] = ps_analysis['spot_decline_pct'] > threshold

cap_by_regime = ps_analysis.groupby('regime')['cap_hit'].agg(['sum','count','mean'])
cap_by_regime.columns = ['cap_hits', 'total_trades', 'cap_hit_rate']

print(f'Short strike at {SHORT_STRIKE_PCT:.0%} of spot — breach at >{threshold:.0%} decline')
print()
print(cap_by_regime.to_string())

In [ ]:
regimes_sorted = cap_by_regime.index.tolist()
rates = (cap_by_regime['cap_hit_rate'] * 100).tolist()

bar_colors = [FID_RED if r > 20 else FID_AMBER if r > 10 else FID_GREEN for r in rates]

fig, ax = plt.subplots(figsize=(12, 5))
fig.text(0.02, 0.98,
         f'Spread Cap Breach Rate by Regime  (SPY decline > {threshold:.0%})',
         fontsize=12, fontweight='bold', va='top')

bars = ax.bar(regimes_sorted, rates, color=bar_colors,
              alpha=0.9, edgecolor=FID_RULE, linewidth=0.5)

for bar, rate in zip(bars, rates):
    ax.text(bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.4,
            f'{rate:.1f}%', ha='center', fontsize=9, color=FID_CHARCOAL)

ax.set_ylabel('Trades Where Short Strike Was Breached (%)')
ax.set_xlabel('Regime at Trade Open')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(fmt_pct))
ax.grid(True, axis='y')
plt.xticks(rotation=30, ha='right')

plt.savefig('../results/05_cap_breach_by_regime.png')
plt.show()

---
## 9. Exit Signal Quality — Reversion Timing

In [ ]:
for label, df_t in [('Long Put', lp_df), ('Put Spread', ps_df)]:
    print(f'{label} — Close Reason Breakdown')
    print('─' * 42)
    s = df_t.groupby('close_reason')['pnl'].agg(['count','sum','mean'])
    s.columns = ['count', 'total_pnl', 'avg_pnl']
    print(s.to_string())
    print()

---
## 10. Summary

In [ ]:
premium_savings = (
    (lp_m['total_premium'] - ps_m['total_premium']) / lp_m['total_premium']
    if lp_m['total_premium'] > 0 else 0
)
pnl_diff = lp_m['total_pnl'] - ps_m['total_pnl']

rows = [
    ('Period',                   f"{BACKTEST_START}  to  {BACKTEST_END}",  ''),
    ('─' * 28,                   '─' * 18,                                 '─' * 18),
    ('',                         'Long Put',                                'Put Spread'),
    ('Total P&L',                f"${lp_m['total_pnl']:>12,.0f}",          f"${ps_m['total_pnl']:>12,.0f}"),
    ('Total Premium',            f"${lp_m['total_premium']:>12,.0f}",       f"${ps_m['total_premium']:>12,.0f}"),
    ('Win Rate',                 f"{lp_m['win_rate']:>12.1%}",              f"{ps_m['win_rate']:>12.1%}"),
    ('Avg Return on Premium',    f"{lp_m['avg_rop']:>12.1%}",               f"{ps_m['avg_rop']:>12.1%}"),
    ('Sharpe (hedge leg)',        f"{lp_m['sharpe']:>12.2f}",               f"{ps_m['sharpe']:>12.2f}"),
    ('─' * 28,                   '─' * 18,                                 '─' * 18),
    ('Premium savings (spread)',  '',                                        f"{premium_savings:.1%}"),
    ('P&L difference (LP - PS)', f"${pnl_diff:>12,.0f}",                   ''),
]

print()
for r in rows:
    print(f'  {r[0]:<28}  {r[1]:<18}  {r[2]}')
print()

---
## Next Steps

1. **Swap in real OptionMetrics data** — set `DATA_MODE = 'optionmetrics'` in `config.py` and populate `data/raw/optionmetrics/`
2. **Strike sensitivity** — adjust `PUT_STRIKE_PCT` and `SHORT_STRIKE_PCT` in `config.py`
3. **DTE sensitivity** — compare 21-DTE vs. 30-DTE vs. 45-DTE rolls
4. **Portfolio overlay** — measure hedge impact on full portfolio Sharpe
5. **Live signal feed** — connect daily regime log for real-time classification